# Customer Segmentation for Online Retail Data

**Customer segmentation is the practice of categorizing a company's customer base into distinct groups with similar characteristics or behaviors**. This allows businesses to better understand and cater to the unique needs and preferences of each group, ultimately leading to improved customer satisfaction, increased sales, and enhanced customer retention.

In this project we will analyze a dataset containing [Online Retail Data](https://www.kaggle.com/datasets/yasserh/customer-segmentation-dataset) from Kaggle. 

## Key Objectives of the Project

- Perform customer segmentation to categorize customers into distinct groups.


- Analyze and understand the characteristics and behaviors of each customer segment.


- Utilize customer segmentation as an opportunity to gain hands-on experience and knowledge in unsupervised learning techniques, particularly K-Means clustering, to effectively group customers based on their shared traits and behaviors

## Importing Libraries 

Let us start by importing the necessary libraries for exploring the data and building our model

In [ ]:
!pip uninstall numpy pandas -y
!pip install numpy pandas

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from scipy.cluster.hierarchy import linkage
from scipy.cluster.hierarchy import dendrogram
from scipy.cluster.hierarchy import cut_tree

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## Read and Explore the Dataset

We'll start by reading and loading this dataset into our Python environment for analysis.

In [ ]:
retail = pd.read_excel('data/Online Retail.xlsx')
retail.head()

In [ ]:
retail.shape

### Understanding the Data

The "Online Retail" dataset is a repository of information related to online retail transactions, including details regarding sold products, quantities, pricing, customer particulars, and transaction timestamps. Its primary application lies in retail analytics, including customer segmentation, sales forecasting, and product performance analysis. The dataset yields valuable insights into customer behavior and online retail sales patterns. Each column within the dataset serves a specific purpose:

- **`InvoiceNo`**: This column presumably holds a unique identifier or code assigned to each individual retail invoice or transaction. It plays an important role in distinguishing and tracing each sales transaction uniquely.


- **`StockCode`**: This column is expected to contain a code or identifier corresponding to the particular product or item sold in each transaction. It facilitates the linkage between products and their respective transactions.


- **`Description`**: This column is likely to encompass textual descriptions or names of the products or items sold, offering more comprehensive insights into the nature of each product.


- **`Quantity`**: This column probably signifies the quantity of the product or item sold during each transaction, denoting the number of units acquired.


- **`InvoiceDate`**`: This column is expected to capture the date and time when each retail invoice or transaction was executed, providing a chronological reference for each sale.


- **`UnitPrice`**`: This column contains the unit price of the product or item being sold, indicating the cost of a single unit.


- **`CustomerID`**`: This column presumably holds a unique identifier or code assigned to each customer who made a purchase, enabling the tracking of individual customer transactions.


- **`Country`**: This column contains the name or code of the country where the purchasing customer is located, offering geographical context regarding customer locations.








In addition, we can explore `info()` and `describe()` methods on the "Online Retail" dataset helps us understand its structure. It shows how many entries we have, what types of data are in each column, in addition to statistical properties of the data. This helps us check if the data is clean and ready for analysis, like customer segmentation.

In [ ]:
retail.info()

In [ ]:
retail.describe()

## Data Cleaning


### Checking for Missing Values

Checking for missing values is a crucial step in data analysis because it ensures the data's quality and integrity. Missing values can significantly impact the reliability of analytical results, potentially leading to biased conclusions or inaccurate predictions.

In [ ]:
# Create a heatmap of missing values
plt.figure(figsize=(10, 6))  # Adjust the figure size as needed
sns.heatmap(retail.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Values Heatmap')
plt.show()

In [ ]:
# Count missing values for each column
missing_values = retail.isnull().sum()
print("Missing Values per Column:")
print(missing_values)

### Dropping Missing Values

The decision to drop rows with missing values is based on the fact that `Description` and `CustomerID` columns have a substantial number of missing entries, These entries cannot be accurately imputed or replaced accurately, so it is best to drop them. By doing so, we ensure that the dataset used for subsequent analyses is free from incomplete or potentially misleading information. In addition the amount of missing values is a relatively small percentage of the entire dataset

In [ ]:
# Droping rows having missing values

retail = retail.dropna()
retail.shape

In addition, we change the data type of "`CustomerID`" to a string because we don't intend to perform numerical operations on this column.

In [ ]:
retail['CustomerID'] = retail['CustomerID'].astype(str)

In [ ]:
# Create Total Revenue column
retail['Revenue'] = retail['Quantity'] * retail['UnitPrice']

# Grouping by CustomerID and aggregating the data
customer_data = retail.groupby('CustomerID').agg({
    'Revenue': 'sum',
    'Quantity': 'sum',
    'InvoiceNo': 'nunique'
}).rename(columns={'InvoiceNo': 'NumInvoices'})

# Calculate average price per invoice
customer_data['AvgPricePerInvoice'] = customer_data['Revenue'] / customer_data['NumInvoices']

customer_data.head()


In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
customer_data_normalized = pd.DataFrame(scaler.fit_transform(customer_data), columns=customer_data.columns)

customer_data_normalized.head()


In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Finding the optimal number of clusters using the Elbow Method
sse = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(customer_data_normalized)
    sse.append(kmeans.inertia_)

# Plotting the results
plt.plot(range(1, 11), sse)
plt.xlabel('Number of clusters')
plt.ylabel('SSE')
plt.title('Elbow Method')
plt.show()


## Exploratory Data Analysis

### Revenue by Country

One specific objective here is to create a visual representation of revenue by country. By doing so, we can identify the countries that contribute the most to our sales revenue and uncover any geographical trends or variations in customer spending patterns. 

In [ ]:
# Create a new column for total revenue (Quantity * UnitPrice)
retail['Revenue'] = retail['Quantity'] * retail['UnitPrice']

# Group the data by 'Country' and calculate total revenue for each country
revenue_by_country = retail.groupby('Country')['Revenue'].sum().reset_index()

# Sort the data by revenue in descending order for better visualization
revenue_by_country = revenue_by_country.sort_values(by='Revenue', ascending=False)

# Create a bar plot to visualize revenue by country
plt.figure(figsize=(12, 6))
plt.barh(revenue_by_country['Country'], revenue_by_country['Revenue'], color='skyblue')
plt.xlabel('Revenue')
plt.ylabel('Country')
plt.title('Revenue by Country')
plt.gca().invert_yaxis()  # Invert y-axis to show the highest revenue at the top
plt.show()

In terms of overall revenue contribution, the majority of revenue originated from the United Kingdom, with the Netherlands and Ireland (EIRE) following closely. This observation aligns logically with the dataset's context, as it primarily comprises transactions from a UK-based online retail business.

### Revenue by Month of the Year

We also provide insights into revenue trends over the months of the year. We begin by converting the `InvoiceDate` column to datetime format and then create a new `YearMonth` column, which allows us to group the data by months. 

Next, we calculate the total revenue for each month by summing the `Revenue` column (which is the result of multiplying `Quantity` and `UnitPrice`). 

The resulting line plot illustrates how revenue fluctuates across different months, providing valuable information for business planning and decision-making.

In [ ]:
retail['InvoiceDate'] = pd.to_datetime(retail['InvoiceDate'])
retail['YearMonth'] = retail['InvoiceDate'].dt.to_period('M')

# Group the data by 'YearMonth' and calculate total revenue for each month
revenue_by_month = retail.groupby('YearMonth')['Revenue'].sum().reset_index()

# Create a bar plot to visualize revenue by month
plt.figure(figsize=(12, 4))
plt.title("Total Revenue by Month of the Year")
sns.barplot(data=revenue_by_month, x="YearMonth", y="Revenue", palette="viridis")
plt.xlabel("Year-Month")
plt.ylabel("Total Revenue")
plt.xticks(rotation=45)  # Rotate x-axis labels for readability
plt.show()


### Revenue by Day of Week

Visualizing revenue by day of the week is important for understanding how sales fluctuate over the course of the week, which can inform staffing, marketing, and inventory management decisions. To achieve this, we'll analyze the `retail` dataset, calculate daily revenue, and create a visual representation of revenue by day of the week.

In [ ]:
retail['InvoiceDate'] = pd.to_datetime(retail['InvoiceDate'])
retail['DayOfWeek'] = retail['InvoiceDate'].dt.day_name()

revenue_by_day = retail.groupby('DayOfWeek')['Revenue'].sum().reset_index()

plt.figure(figsize=(10, 5))
plt.title("Revenue by Day of the Week")
sns.barplot(data=revenue_by_day, x="DayOfWeek", y="Revenue", order=["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"], palette="viridis")
plt.xlabel("Day of the Week")
plt.ylabel("Total Revenue")
plt.show()


Thursday emerges as the day with the highest revenue in our analysis, followed by Tuesday. However, it's worth noting that Saturday seems to lack revenue data entirely. This led us to explore the dataset further to ensure data accuracy and confirm whether there were indeed no transactions recorded on Saturdays.

In [ ]:
# Filter data for Saturdays
saturdays_data = retail[retail['InvoiceDate'].dt.day_name() == 'Saturday']

# Check the number of transactions and total revenue on Saturdays
print("Number of Saturday Transactions:", len(saturdays_data))
print("Total Saturday Revenue:", saturdays_data['Revenue'].sum())


### Top 10 Customers by Revenue

Identifying the top-performing customers by revenue is crucial for businesses looking to optimize their customer-focused strategies. In our analysis, we have generated a list of the top 10 customers who have contributed the highest revenue. This information can be instrumental in tailoring marketing efforts, enhancing customer relationships, and maximizing revenue generation from key clientele.

In [ ]:
# Group the data by 'CustomerID' and calculate total revenue for each customer
revenue_by_customer = retail.groupby('CustomerID')['Revenue'].sum().reset_index()

# Sort the data by revenue in descending order
top_10_customers = revenue_by_customer.sort_values(by='Revenue', ascending=False).head(10)

# Display the top 10 customers by revenue
print("Top 10 Customers by Revenue:")
print(top_10_customers)

### Analyze Top Selling Products

Our goal is to identify and visualize the top-selling products in the "retail" dataset based on two key metrics: quantity sold and revenue generated. We calculate the total quantity and total revenue for each product, sorts them in descending order, and selects the top 10 products for each metric. The code then visualizes these top-selling products using bar plots, providing a clear understanding of which products are performing best in terms of both quantity and revenue.

In [ ]:
# Top-Selling Products by Quantity
top_products_quantity = retail.groupby('Description')['Quantity'].sum().reset_index()
top_products_quantity = top_products_quantity.sort_values(by='Quantity', ascending=False).head(10)

# Top-Selling Products by Revenue
top_products_revenue = retail.groupby('Description')['Revenue'].sum().reset_index()
top_products_revenue = top_products_revenue.sort_values(by='Revenue', ascending=False).head(10)

# Visualize the top-selling products by quantity
plt.figure(figsize=(12, 6))
plt.title("Top-Selling Products by Quantity")
sns.barplot(data=top_products_quantity, y='Description', x='Quantity', palette='viridis')
plt.xlabel("Quantity")
plt.ylabel("Product Description")
plt.show()

# Visualize the top-selling products by revenue
plt.figure(figsize=(12, 6))
plt.title("Top-Selling Products by Revenue")
sns.barplot(data=top_products_revenue, y='Description', x='Revenue', palette='viridis')
plt.xlabel("Revenue")
plt.ylabel("Product Description")
plt.show()

## Clustering: Customer Segmentation

The goal of clustering and customer segmentation in our analysis is to group customers into categories based on their shopping behavior, specifically looking at how much they buy, how much they spend, and their sensitivity to prices. This helps us better understand our customers and allows us to customize our marketing and product strategies to meet the unique needs of each group. By identifying these customer segments, we can make data-driven decisions to boost sales and keep our customers happy.

### Data Normalization

Data normalization is essential because it ensures that all the numerical features in our dataset have the same scale, preventing certain features from dominating the clustering process solely because they have larger values. This is particularly important for algorithms like K-means, which are sensitive to feature scales. Normalizing the data transforms it into a standardized form, where the mean is 0 and the standard deviation is 1 for each feature, allowing for fair and meaningful comparisons.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Select the numeric features you want to normalize (e.g., Quantity, UnitPrice, Revenue)
numeric_features = ['Quantity', 'UnitPrice', 'Revenue']

# Initialize the MinMaxScaler
scaler = MinMaxScaler()

# Fit and transform the selected features
retail[numeric_features] = scaler.fit_transform(retail[numeric_features])

In [ ]:
retail.head()

### Choose the Correct Number of Clusters

Choosing the correct number of clusters is crucial in customer segmentation because it defines the granularity and interpretability of your segments. An inappropriate number of clusters can lead to either overly broad or overly detailed segments, making it challenging to derive actionable insights. To select the optimal number of clusters, we can use techniques like the silhouette score. The silhouette score quantifies the similarity of data points within a cluster compared to data points in other clusters. A higher silhouette score indicates better-defined clusters. By choosing the number of clusters that maximizes the silhouette score, we ensure that our segmentation is both meaningful and useful for tailoring marketing strategies, optimizing product offerings, and enhancing customer engagement for distinct customer groups.

In [ ]:
retail['Revenue'] = retail['Quantity'] * retail['UnitPrice']
retail.head()

In [ ]:
customer_data = retail.groupby('CustomerID').agg({
    'Quantity': 'sum',
    'Revenue': 'sum',
    'InvoiceNo': 'nunique',
    'InvoiceDate': lambda x: (retail['InvoiceDate'].max() - x.max()).days
}).rename(columns={'InvoiceNo': 'Frequency', 'InvoiceDate': 'Recency'})


In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Create a range of possible cluster numbers
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=42)
    kmeans.fit(customer_data)
    wcss.append(kmeans.inertia_)

# Plot the results
plt.plot(range(1, 11), wcss)
plt.title('Elbow Method')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.show()
